In [11]:
import earthaccess
import pandas as pd
import xarray as xr
import h5py
import io

# 1. Autenticação
auth = earthaccess.login(strategy="interactive")

def investigar_conteudo_datasets():
    datasets_alvo = [
        "IPANC1B", "ILATM1B", "ILATM2", "IRMCR1B", 
        "IPFLR1B", "IAVAPS1B", "IGGRV1B", "IPFLT1B"
    ]
    
    print(f"🔬 Iniciando Investigação Técnica em {len(datasets_alvo)} datasets...\n")
    
    for sn in datasets_alvo:
        print(f"\n{'='*100}")
        print(f"📂 DATASET: {sn}")
        
        # --- PARTE 1: O que é este dataset? (Metadata da Coleção) ---
        colecao = earthaccess.search_datasets(short_name=sn)
        if colecao:
            abstract = colecao[0].abstract()
            print(f"\n📝 RESUMO CIENTÍFICO:\n{abstract[:500]}...") 
        
        # --- PARTE 2: O que tem dentro do arquivo? (Exploração de Granule) ---
        resultados = earthaccess.search_data(
            short_name=sn,
            bounding_box=(-180, -90, 180, -60),
            count=1 
        )
        
        if not resultados:
            print(f"\n⚠️  STATUS: Sem dados na região Antártica.")
            continue
            
        granule = resultados[0]
        links = [l['URL'] for l in granule['umm']['RelatedUrls'] if 'GET DATA' in str(l.get('Type'))]
        extensao = links[0].split('.')[-1].upper() if links else "N/A"
        
        print(f"\n✅ ARQUIVO ENCONTRADO: {links[0].split('/')[-1]}")
        print(f"📁 FORMATO: {extensao}")

        # --- PARTE 3: Inspeção Remota de Variáveis ---
        print("\n🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):")
        try:
            files = earthaccess.open([granule])
            with files[0] as f:
                # Caso A: Arquivos Binários (HDF5 / NetCDF)
                if extensao in ['H5', 'HDF5', 'NC', 'NC4']:
                    with h5py.File(f, 'r') as h5_file:
                        def print_structure(name, obj):
                            if isinstance(obj, h5py.Dataset):
                                print(f"  - Variável: {name} | Shape: {obj.shape} | Tipo: {obj.dtype}")
                        h5_file.visititems(print_structure)
                
                # Caso B: Arquivos de Texto (CSV / TXT)
                elif extensao in ['CSV', 'TXT', 'QI']:
                    head = f.read(1000).decode('utf-8')
                    print("  - [Amostra de Texto/Cabeçalho]:")
                    print(f"\n{head}")
                
                # Caso C: Relatórios (PDF)
                elif extensao == 'PDF':
                    print("  - [Documento PDF]: Contém logs operacionais legíveis por humanos.")

        except Exception as e:
            print(f"❌ Erro ao espiar variáveis: {e}")

if __name__ == "__main__":
    investigar_conteudo_datasets()

🔬 Iniciando Investigação Técnica em 8 datasets...


📂 DATASET: IPANC1B

⚠️  STATUS: Sem dados na região Antártica.

📂 DATASET: ILATM1B

📝 RESUMO CIENTÍFICO:
This data set contains spot elevation measurements of Arctic and Antarctic sea ice, and Greenland, Antarctic Peninsula, and West Antarctic region ice surface acquired using the NASA Airborne Topographic Mapper (ATM) instrumentation. The data were collected as part of Operation IceBridge funded aircraft survey campaigns....

✅ ARQUIVO ENCONTRADO: ILATM1B_20091016_164318.atm4cT3.qi
📁 FORMATO: QI

🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

❌ Erro ao espiar variáveis: 'EarthAccessFile' object does not support the context manager protocol

📂 DATASET: ILATM2

📝 RESUMO CIENTÍFICO:
This data set contains resampled and smoothed elevation measurements of Arctic and Antarctic sea ice, and Greenland, Antarctic Peninsula, and West Antarctic region land ice surface acquired using the NASA Airborne Topographic Mapper (ATM) instrumentation. The data were collected as part of Operation IceBridge funded aircraft survey campaigns....

✅ ARQUIVO ENCONTRADO: ILATM2_20091016_164318_smooth_nadir5seg_50pt.csv
📁 FORMATO: CSV

🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):


...

❌ Erro ao espiar variáveis: 'EarthAccessFile' object does not support the context manager protocol

📂 DATASET: IPFLT1B

📝 RESUMO CIENTÍFICO:
This data set contains simplified, or thinned, flight lines from NASA Operation IceBridge Greenland, Arctic, Antarctic, and Alaska missions. The thinning was performed using a Python library called Shapely. The full resolution flight line data were collected as part of Operation IceBridge funded aircraft survey campaigns.

The corresponding flight reports can be found in the IceBridge L1B Flight Reports (IPFLR1B) data set....

✅ ARQUIVO ENCONTRADO: IPFLT1B_20091016_121052.dbf
📁 FORMATO: DBF

🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

❌ Erro ao espiar variáveis: 'EarthAccessFile' object does not support the context manager protocol
